# Training a Small Language Model from Scratch

Decoder-only transformer trained on TinyStories using PyTorch.

**Prerequisites:** Upload the full repo to Colab (or mount via Google Drive).  
Required files: `src/` directory, `dataset/tokenizer.json`, `dataset/tokens_parts/`.

In [ ]:
# PyTorch is pre-installed on Colab — just verify GPU
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
PATH_TO_REPO = "/content/vSLM-From-Scratch/"

import sys
sys.path.insert(0, PATH_TO_REPO)

In [ ]:
from src.model.config import ModelConfig
from src.model.transformer import Transformer
from src.data.dataset import TokenDataset
from src.training.trainer import Trainer, TrainingConfig
from src.tokenization.tokenizer import BPETokenizer

from pathlib import Path

## Google Drive Checkpoints

Mount Google Drive to persist checkpoints across Colab sessions.  
Set `GDRIVE_CKPT_DIR` to the folder where checkpoints will be saved/loaded.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

GDRIVE_CKPT_DIR = Path("/content/drive/MyDrive/vSLM-checkpoints")
GDRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint dir: {GDRIVE_CKPT_DIR}")

In [ ]:
# Find the latest checkpoint on GDrive (if any) for resuming
def find_latest_checkpoint(ckpt_dir):
    """Return path to the most recent checkpoint, or None."""
    ckpts = sorted(ckpt_dir.glob("step_*.pt"))
    if ckpts:
        return ckpts[-1]  # highest step number (lexicographic sort works for zero-padded)
    # Also check for a final.pt
    final = ckpt_dir / "final.pt"
    return final if final.exists() else None

latest_ckpt = find_latest_checkpoint(GDRIVE_CKPT_DIR)
if latest_ckpt:
    print(f"Found checkpoint to resume from: {latest_ckpt}")
else:
    print("No existing checkpoints found — will train from scratch")

## 1. Load Tokenizer

In [ ]:
tokenizer = BPETokenizer.load(PATH_TO_REPO + "dataset/tokenizer.json")
print(f"Vocab size: {len(tokenizer.vocab)}")

## 2. Load Token Data

Loads all `part_XXXXX.npy` files from `tokens_parts/` and concatenates them.  
Use `max_parts=N` to limit for quick experiments.

In [ ]:
CONTEXT_LENGTH = 128

dataset = TokenDataset.from_parts(
    PATH_TO_REPO + "dataset/tokens_parts",
    context_length=CONTEXT_LENGTH,
    # max_parts=50,  # uncomment to use fewer parts for a quick test
)
train_data, val_data = dataset.split(val_fraction=0.05)

In [ ]:
# Verify: inspect a sample batch
x, y = train_data.get_batch(batch_size=4, device="cpu")
print(f"Input shape:  {x.shape}")
print(f"Target shape: {y.shape}")
print(f"\nInput[0][:10]:  {x[0][:10].tolist()}")
print(f"Target[0][:10]: {y[0][:10].tolist()}")
print("(target should be input shifted right by 1)")

# Decode a sample to see actual text
print(f"\nDecoded input[0]: {tokenizer.decode(x[0].tolist())[:200]}...")

## 3. Configure Model

Default: ~5M params. Scale up by increasing `d_model`, `n_layers`, `n_heads`.

In [ ]:
model_config = ModelConfig(
    vocab_size=len(tokenizer.vocab),  # 4096
    context_length=CONTEXT_LENGTH,    # 128
    d_model=256,
    n_heads=4,
    n_layers=4,
    d_ff=1024,
    dropout_rate=0.1,
)

model = Transformer(model_config)
print(model_config)
print(f"Parameters: {model.count_parameters():,}")

## 4. Configure Training

In [ ]:
train_config = TrainingConfig(
    batch_size=256,              # ← bigger batch to fill GPU (was 64)
    learning_rate=6e-4,          # ← scale LR with batch size
    warmup_steps=500,
    max_steps=50_000,
    weight_decay=0.1,
    grad_clip=1.0,
    log_every=100,
    eval_every=2000,
    eval_batches=20,
    checkpoint_every=10_000,
    checkpoint_dir=str(GDRIVE_CKPT_DIR),  # ← save to Google Drive
    seed=42,
    use_amp=True,                # FP16 mixed precision
    compile_model=True,          # torch.compile fused kernels
)

trainer = Trainer(model, model_config, train_config)

## 5. Train

In [ ]:
history = trainer.train(train_data, val_data, resume_from=latest_ckpt)

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["step"], history["train_loss"])
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")

if history["val_loss"]:
    eval_steps = [s for s in history["step"] if s % train_config.eval_every == 0]
    axes[1].plot(eval_steps[: len(history["val_loss"])], history["val_loss"])
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Validation Loss")

plt.tight_layout()
plt.show()

## 7. Generate Text

In [ ]:
prompts = [
    "Once upon a time",
    "The little dog",
    "She was very happy because",
]

for prompt in prompts:
    text = trainer.generate(
        tokenizer, prompt,
        max_tokens=200, temperature=0.8, top_k=40,
    )
    print(f"--- Prompt: {prompt!r} ---")
    print(text)
    print()

In [ ]:
# Interactive: try your own prompt
text = trainer.generate(
    tokenizer,
    prompt="There was a big",
    max_tokens=300,
    temperature=0.7,
    top_k=50,
)
print(text)